# 11 — Band-pass filters (`math`, SciPy, PyTorch, python-control)

**Baseline:** the analog prototype

$$
H(s) = \frac{2\zeta\omega_n s}{s^2 + 2\zeta\omega_n s + \omega_n^2}
$$

centered at \(f_n\) with damping \(\zeta\). A square wave at \(f_n\) is mostly the fundamental after this filter; a square far outside the band is rejected.

Same four stacks as notebook `10`.


In [ ]:
from __future__ import annotations

import math

import control as ct
import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy import signal

%matplotlib inline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"python-control {ct.__version__}   torch {torch.__version__}   device {device}")


In [ ]:
FS_HZ = 8_192
DURATION_S = 0.12
FN_HZ = 220.0              # band center = tone we want to keep
ZETA = 0.15                # modest Q ≈ 1/(2ζ) ≈ 3.3
F_INBAND = 220.0
F_OUTBAND = 880.0          # two octaves up — should be attenuated
N = int(FS_HZ * DURATION_S)
DT = 1.0 / FS_HZ
WN = 2.0 * math.pi * FN_HZ

t = np.arange(N, dtype=np.float64) / FS_HZ
u_keep = signal.square(2.0 * np.pi * F_INBAND * t)
u_reject = signal.square(2.0 * np.pi * F_OUTBAND * t)

print(f"fn = {FN_HZ} Hz   ζ = {ZETA}   Q ≈ {1.0 / (2.0 * ZETA):.2f}")


## python-control — \(H(s)\), Bode, two forced responses


In [ ]:
num = [2.0 * ZETA * WN, 0.0]
den = [1.0, 2.0 * ZETA * WN, WN**2]
sys_bp = ct.tf(num, den)
print(sys_bp)

t_k, y_ct_keep = ct.forced_response(sys_bp, T=t, U=u_keep)
_, y_ct_rej = ct.forced_response(sys_bp, T=t, U=u_reject)

frd = ct.frequency_response(sys_bp, omega=np.logspace(2, 5, 500))
if hasattr(frd, "magnitude"):
    w = np.asarray(frd.frequency).reshape(-1)
    m = np.asarray(frd.magnitude).reshape(-1)
    p = np.asarray(frd.phase).reshape(-1)
else:
    mag, phase, omega = frd
    w = np.asarray(omega).reshape(-1)
    m = np.asarray(mag).reshape(-1)
    p = np.asarray(phase).reshape(-1)

fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
axes[0].semilogx(w / (2 * np.pi), 20 * np.log10(np.maximum(m, 1e-16)))
axes[0].axvline(FN_HZ, color="k", ls="--", lw=1, label=f"fn = {FN_HZ} Hz")
axes[0].axvline(F_OUTBAND, color="0.5", ls=":", lw=1, label=f"reject = {F_OUTBAND} Hz")
axes[0].set_ylabel("mag (dB)")
axes[0].grid(True, which="both", alpha=0.3)
axes[0].legend()
axes[1].semilogx(w / (2 * np.pi), np.degrees(p))
axes[1].set_ylabel("phase (deg)")
axes[1].set_xlabel("frequency (Hz)")
axes[1].grid(True, which="both", alpha=0.3)
fig.suptitle("python-control Bode — second-order band-pass")
fig.tight_layout()
plt.show()


## `math` — discretized state-space (forward Euler)

$$
\dot{x}_1 = x_2, \qquad
\dot{x}_2 = -\omega_n^2 x_1 - 2\zeta\omega_n x_2 + u, \qquad
y = 2\zeta\omega_n x_2.
$$

This is the controllable-canonical realization of \(H(s)\) above.


In [ ]:
def math_bandpass(u_list: list[float], dt: float, wn: float, zeta: float) -> list[float]:
    x1 = 0.0
    x2 = 0.0
    two_z_wn = 2.0 * zeta * wn
    wn2 = wn * wn
    out = []
    for un in u_list:
        dx1 = x2
        dx2 = -wn2 * x1 - two_z_wn * x2 + un
        x1 = x1 + dt * dx1
        x2 = x2 + dt * dx2
        out.append(two_z_wn * x2)
    return out


y_math_keep = math_bandpass(u_keep.tolist(), DT, WN, ZETA)
y_math_rej = math_bandpass(u_reject.tolist(), DT, WN, ZETA)
print(f"math in-band last={y_math_keep[-1]:.4f}  out-of-band last={y_math_rej[-1]:.4f}")


## SciPy — Butterworth band-pass around \(f_n\)

A first-order-per-side Butterworth (`N=1`) is the discrete cousin of the analog prototype. Band edges are placed at \(f_n / \sqrt{2}\) and \(f_n\sqrt{2}\) (one octave wide).


In [ ]:
lo = FN_HZ / math.sqrt(2.0)
hi = FN_HZ * math.sqrt(2.0)
b_sp, a_sp = signal.butter(N=1, Wn=(lo, hi), btype="bandpass", fs=FS_HZ)
y_sp_keep = signal.lfilter(b_sp, a_sp, u_keep)
y_sp_rej = signal.lfilter(b_sp, a_sp, u_reject)
print("band edges (Hz):", lo, hi)
print("b =", b_sp)
print("a =", a_sp)


## PyTorch — Euler state update on device + FIR band-pass `conv1d`


In [ ]:
def torch_bandpass_euler(u_t: torch.Tensor, dt: float, wn: float, zeta: float) -> torch.Tensor:
    x1 = torch.zeros((), device=u_t.device, dtype=u_t.dtype)
    x2 = torch.zeros((), device=u_t.device, dtype=u_t.dtype)
    two_z_wn = 2.0 * zeta * wn
    wn2 = wn * wn
    out = torch.empty_like(u_t)
    for i in range(u_t.numel()):
        un = u_t[i]
        dx1 = x2
        dx2 = -wn2 * x1 - two_z_wn * x2 + un
        x1 = x1 + dt * dx1
        x2 = x2 + dt * dx2
        out[i] = two_z_wn * x2
    return out


def torch_fir_bandpass(u_t: torch.Tensor, f_lo: float, f_hi: float, fs: float, taps: int = 257) -> torch.Tensor:
    if taps % 2 == 0:
        raise ValueError("odd tap count required")
    n = torch.arange(taps, device=u_t.device, dtype=u_t.dtype) - (taps - 1) / 2
    h_hi = 2 * (f_hi / fs) * torch.sinc(2 * (f_hi / fs) * n)
    h_lo = 2 * (f_lo / fs) * torch.sinc(2 * (f_lo / fs) * n)
    h = h_hi - h_lo
    h = h * torch.hamming_window(taps, periodic=False, device=u_t.device, dtype=u_t.dtype)
    # unity gain at band center via a discrete-time cosine probe
    f_c = 0.5 * (f_lo + f_hi)
    probe = torch.cos(2 * math.pi * f_c * n / fs)
    gain = (h * probe).sum()
    h = h / gain
    return torch.nn.functional.conv1d(
        u_t.view(1, 1, -1), h.view(1, 1, -1), padding=taps // 2
    ).view(-1)


u_keep_th = torch.as_tensor(u_keep, device=device, dtype=torch.float32)
u_rej_th = torch.as_tensor(u_reject, device=device, dtype=torch.float32)

y_th_keep = torch_bandpass_euler(u_keep_th, DT, WN, ZETA).detach().cpu().numpy()
y_th_rej = torch_bandpass_euler(u_rej_th, DT, WN, ZETA).detach().cpu().numpy()
y_fir_keep = torch_fir_bandpass(u_keep_th, lo, hi, FS_HZ).detach().cpu().numpy()
y_fir_rej = torch_fir_bandpass(u_rej_th, lo, hi, FS_HZ).detach().cpu().numpy()
print("torch device", u_keep_th.device)


## Overlay — keep \(220\,\mathrm{Hz}\), reject \(880\,\mathrm{Hz}\)


In [ ]:
ms = t * 1e3
fig, axes = plt.subplots(2, 1, figsize=(10, 6.5), sharex=True)

axes[0].plot(ms, u_keep, color="0.8", lw=0.8, label="square 220 Hz")
axes[0].plot(ms, y_ct_keep, lw=2.0, label="python-control")
axes[0].plot(ms, y_math_keep, ls="--", label="math Euler")
axes[0].plot(ms, y_sp_keep, ls="-.", label="scipy butter")
axes[0].plot(ms, y_th_keep, ls=":", label=f"torch Euler ({device})")
axes[0].plot(ms, y_fir_keep, lw=1.0, alpha=0.85, label="torch FIR")
axes[0].set_ylabel("in-band")
axes[0].grid(True, alpha=0.3)
axes[0].legend(ncol=3, fontsize=8)

axes[1].plot(ms, u_reject, color="0.8", lw=0.8, label="square 880 Hz")
axes[1].plot(ms, y_ct_rej, lw=2.0, label="python-control")
axes[1].plot(ms, y_math_rej, ls="--", label="math Euler")
axes[1].plot(ms, y_sp_rej, ls="-.", label="scipy butter")
axes[1].plot(ms, y_th_rej, ls=":", label=f"torch Euler ({device})")
axes[1].plot(ms, y_fir_rej, lw=1.0, alpha=0.85, label="torch FIR")
axes[1].set_ylabel("out-of-band")
axes[1].set_xlabel("time (ms)")
axes[1].grid(True, alpha=0.3)

fig.suptitle(f"band-pass  fn = {FN_HZ} Hz  ζ = {ZETA}")
fig.tight_layout()
plt.show()


def rms(x) -> float:
    a = np.asarray(x, dtype=np.float64)
    return float(np.sqrt(np.mean(a * a)))


print(f"{'stack':<18} {'rms keep':>10} {'rms reject':>12} {'reject/keep':>12}")
for name, yk, yr in [
    ("python-control", y_ct_keep, y_ct_rej),
    ("math Euler", y_math_keep, y_math_rej),
    ("scipy butter", y_sp_keep, y_sp_rej),
    ("torch Euler", y_th_keep, y_th_rej),
    ("torch FIR", y_fir_keep, y_fir_rej),
]:
    rk, rr = rms(yk), rms(yr)
    print(f"{name:<18} {rk:10.4f} {rr:12.4f} {rr / max(rk, 1e-12):12.3f}")


## Takeaways

- python-control writes the analog spec once; Bode tells you the keep/reject bands before you discretize.
- Forward-Euler `math` / PyTorch tracks that spec if \(f_s \gg f_n\). At \(8\,\mathrm{kHz}\) and \(220\,\mathrm{Hz}\) that is true; it will drift near Nyquist.
- SciPy Butterworth and the analog prototype share a *shape*, not identical coefficients — compare RMS ratios, not sample-wise equality.
- `conv1d` FIR is the GPU-native form; the Euler loop is there to show the identical state update on `cuda` tensors.
- Later decades (`20+`) can swap these filters for identified models, LQR, or learned controllers without changing the signal / filter contract from `00`–`11`.
